# RSNA Knee: сборка «сырого» кэша на Kaggle (билдер)

## Зачем этот ноутбук

Kaggle даёт МРТ колен в формате DICOM: около 100 тысяч мелких файлов `.dcm`, у каждого аппарата свой
поворот, масштаб и яркость. Учить на этом напрямую медленно и неудобно, а скачать к себе на Mac всё это
нельзя (сотни гигабайт). Билдер **один раз** превращает этот хаос в аккуратные стопки картинок:
**по одному файлу `studies/<ID>.npz` на колено**, внутри - все срезы каждой выбранной серии по порядку,
в одном масштабе и одной ориентации. Это и есть «сырой» кэш, на котором учится `02_report_labels_mil.ipynb`.

## Матрёшка данных

| Уровень | В жизни | В данных соревнования | После билдера |
|---|---|---|---|
| исследование | один визит пациента, одно колено | папка `train_series/<ID колена>/` | файл `studies/<ID>.npz` |
| серия | одна «нарезка» колена с одной стороны в одном режиме | папка `<ID колена>/<ID серии>/`, 3-14 штук на колено | ключ `slot0`...`slot5` в файле, не больше 6 |
| срез | один ломтик, одна картинка | один файл `.dcm`, имена случайные | одна картинка 224×224 в массиве `[n, 224, 224]`, по порядку |

## Что делает билдер и почему именно здесь

Здесь только то, для чего нужны заголовки DICOM («паспорта» срезов). Самих DICOM локально не будет,
поэтому позже этого уже не сделать:

| Шаг | Как | Зачем |
|---|---|---|
| выбор серий | 6 ящиков = 3 плоскости × (с подавлением жира / без); в каждый ящик серия, где больше срезов | у колена 3-14 серий, многие повторяются |
| порядок срезов | по координате среза в теле (`ImagePositionPatient`) вдоль направления нарезки (`ImageOrientationPatient`) | имена файлов случайные |
| единая ориентация | транспонировать / отразить по `ImageOrientationPatient` | разные аппараты кладут картинку по-разному |
| кроп 140 мм и 224 px | по размеру пикселя в мм (`PixelSpacing`) | у всех колен один физический масштаб |
| яркость 0-255 | перцентили 0.5% и 99.5% серии -> 0 и 255 | у МРТ нет единой шкалы яркости; uint8 = 1 байт на пиксель |

Чего здесь **нет** (это делает класс `Dataset` при обучении): заполнение пустых ящиков, выбор окон, зеркало,
аугментации, нормализация ImageNet. Причина: это либо случайное (должно меняться каждую эпоху), либо
настройки, которые хочется менять без пересборки кэша.

## Две части

Kaggle сохраняет в Output не больше 20 ГБ за запуск, а все срезы весят ~22 ГБ. Поэтому колена делятся на
`N_PARTS = 2` части (каждое второе по порядку ID), и ноутбук запускается дважды: с `PART = 0` и с `PART = 1`.
Каждый запуск становится отдельным датасетом.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл. Input: соревнование `rsna-knee-abnormality-detection`.
2. Settings: Accelerator **None** (CPU), Internet можно выключить. CPU не тратит недельную квоту GPU.
3. `PART = 0` -> **Save Version -> Save & Run All**. Когда досчитается: **Output -> New Dataset** `rsna-knee-raw-part0`.
4. **Copy & Edit**, `PART = 1`, снова **Save & Run All**, потом **New Dataset** `rsna-knee-raw-part1`
   (из одного ноутбука Kaggle разрешает сделать только один датасет).
5. Каждую часть упаковать в один файл ноутбуком `05_pack_cache_kaggle.ipynb` и скачать.

## Что получится и куда положить локально

```
data/raw/
├── studies/<StudyInstanceUID>.npz     4407 файлов из обеих частей
├── index_part0.csv, index_part1.csv   какая серия в каком ящике, сколько в ней срезов
├── rsna_raw_meta.json                 настройки сборки (IMG, CROP_MM, MAX_SLICES, слоты)
└── train.csv                          метки и тексты заключений
```

Обе части распаковываются в одну папку `data/raw/`: папки `studies/` сольются.

## Импорты

| Библиотека | Зачем здесь |
|---|---|
| `os`, `glob`, `pathlib.Path` | пути к файлам, список файлов в папке |
| `concurrent.futures.ThreadPoolExecutor` | читать много серий одновременно (диск Kaggle медленный, потоки ждут его параллельно) |
| `pydicom` | читать файлы `.dcm`: заголовок («паспорт» среза) и картинку |
| `cv2` (OpenCV) | быстрый ресайз картинок |
| `numpy`, `pandas` | массивы и таблицы |
| `matplotlib` | картинки для проверки глазами |
| `torch`, `timm`, `sklearn` | здесь не нужны, остались от общего списка импортов |

`OMP_NUM_THREADS` и похожие: сколько потоков могут брать математические библиотеки. Ограничиваем 4, чтобы
24 потока чтения не дрались с ними за процессор.

In [ ]:
import os
# сколько потоков могут занимать математические библиотеки (numpy, OpenCV); setdefault - если не задано раньше
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import re
import math
import time
import glob
import json
import hashlib
import textwrap
import warnings
from copy import deepcopy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor     # пул потоков: много серий читаются одновременно

import cv2                                             # ресайз картинок
import numpy as np
import pandas as pd
import pydicom                                         # чтение файлов .dcm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")                      # не засорять вывод предупреждениями библиотек
T0 = time.time()                                       # время старта: log() печатает секунды от него

def log(msg):
    # печать с отметкой времени: [  12.3s] сообщение. flush=True - сразу, а не когда буфер заполнится
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display                                            # в Jupyter display красиво рисует таблицы
except NameError:
    display = print                                    # вне Jupyter - обычный print

# Устройство: видеокарта NVIDIA (Kaggle) -> видеокарта Apple Silicon (MPS) -> процессор
DEV = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


def empty_device_cache():
    # освободить кэш памяти видеокарты; у CUDA и MPS это разные функции
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    elif DEV.type == "mps":
        torch.mps.empty_cache()
log(f"device: {DEV}, torch {torch.__version__}, timm {timm.__version__}, pydicom {pydicom.__version__}")

## Настройки

Между двумя запусками меняется только `PART`. Остальное должно совпадать в обоих запусках, иначе части
кэша будут разными по формату.

| Параметр | Значение | Что значит |
|---|---|---|
| `N_PARTS` | 2 | на сколько запусков делим колена (лимит Output 20 ГБ за запуск) |
| `PART` | 0 или 1 | какую часть собираем сейчас |
| `IMG` | 224 | сторона среза в пикселях после ресайза. 224 - родной размер для ViT с патчем 14 (16×16 патчей) |
| `CROP_MM` | 140 | какой квадрат колена вырезаем, в миллиметрах. 140 мм с запасом вмещает сустав |
| `MAX_SLICES` | 32 | больше срезов из серии не берём (у большинства серий 18-35) |
| `SERIES_THREADS` | 24 | сколько серий читаем одновременно |
| `CHUNK_STUDIES` | 64 | сколько колен обрабатываем за раз: держим в памяти только их |

Один пиксель результата = 140 / 224 = **0.625 мм** у всех колен.

In [ ]:
# Пути Kaggle: данные соревнования лежат в одном из двух мест (зависит от версии Kaggle)
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/rsna-knee-abnormality-detection")
OUT_DIR = Path("/kaggle/working")       # всё, что здесь, станет содержимым датасета
STUDY_DIR = OUT_DIR / "studies"         # по файлу на исследование
STUDY_DIR.mkdir(parents=True, exist_ok=True)   # parents=True - создать и родительские папки; exist_ok - не падать, если есть

N_PARTS = 2                # на сколько запусков делим исследования (лимит Output 20 ГБ)
PART = 0                   # какую часть собираем сейчас: 0 или 1

IMG = 224                  # сторона среза в пикселях
CROP_MM = 140.0            # квадрат 140x140 мм вокруг центра снимка -> один физический масштаб у всех
MAX_SLICES = 32            # если в серии больше срезов, берём 32 равномерно (у большинства серий 18-35)
SERIES_THREADS = 24        # сколько серий читаем параллельно (упор в диск Kaggle, а не в процессор)
CHUNK_STUDIES = 64         # сколько исследований обрабатываем за раз (ограничивает память)

log(f"DATA_DIR = {DATA_DIR} (существует: {DATA_DIR.exists()}), часть {PART} из {N_PARTS}")

## Что лежит в данных соревнования

Прежде чем что-то собирать, посмотрим на сырьё:

- `train_series.csv`: **одна строка = одна серия**. Колонки `StudyInstanceUID` (какое колено), `SeriesInstanceUID`
  (какая серия), `Anatomical_Plane` (с какой стороны резали), `Fat_Suppression` (1 = с подавлением жира);
- папка одной серии: файлы `.dcm` со случайными именами, по одному на срез;
- заголовок одного `.dcm`: те самые 3 поля, по которым билдер наводит порядок.

| Поле заголовка | Пример | Что значит |
|---|---|---|
| `ImagePositionPatient` | `[-62.1, -95.3, 41.8]` | где в теле лежит левый верхний угол среза, мм (x - к левому боку, y - к спине, z - к голове) |
| `ImageOrientationPatient` | `[0, 1, 0, 0, 0, -1]` | 6 чисел: куда в теле идут строки картинки (первые 3) и столбцы (вторые 3) |
| `PixelSpacing` | `[0.45, 0.45]` | сколько мм в одном пикселе по вертикали и горизонтали |
| `InstanceNumber` | `14` | номер среза от сканера (бывает ненадёжным, поэтому порядок берём по координатам) |

In [ ]:
series_demo = pd.read_csv(DATA_DIR / "train_series.csv")       # одна строка = одна серия
print("train_series.csv:", series_demo.shape, "колонки:", list(series_demo.columns))
display(series_demo.head())
per_study = series_demo.groupby("StudyInstanceUID").size()       # сколько серий у каждого колена
print(f"серий на колено: от {per_study.min()} до {per_study.max()}, в среднем {per_study.mean():.1f}")
display(series_demo.groupby(["Anatomical_Plane", "Fat_Suppression"]).size().rename("серий").to_frame())

# папка одной серии: что в ней лежит
r0 = series_demo.iloc[0]
d0 = DATA_DIR / "train_series" / str(r0["StudyInstanceUID"]) / str(r0["SeriesInstanceUID"])
files0 = sorted(f for f in os.listdir(d0) if f.endswith(".dcm"))
print(f"\nсерия {r0['Anatomical_Plane']}, Fat_Suppression={r0['Fat_Suppression']}: {len(files0)} файлов .dcm, "
      f"первые имена: {files0[:4]}")

# заголовок одного файла: stop_before_pixels=True - читаем только «паспорт», без картинки
h0 = pydicom.dcmread(d0 / files0[0], stop_before_pixels=True)
for tag in ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber", "Rows", "Columns"]:
    print(f"  {tag:<24} {getattr(h0, tag, None)}")

## Шаг 1. `pick_series`: какую серию в какой ящик

У колена 3-14 серий, а ящиков 6 (3 плоскости × 2 режима). Многие серии повторяют одну и ту же клетку
(тот же угол и режим, другие настройки сканера или пересъёмка). Правило простое:

1. для каждой серии считаем файлы `.dcm` в её папке (`count_files`) = сколько в ней срезов;
2. для каждого колена и каждого из 6 ящиков берём серии нужной плоскости и режима, где **не меньше 3 срезов**
   (1-2 среза - это обзорные «прицельные» снимки);
3. из них берём серию с **наибольшим числом срезов**. Нет подходящих - ящик пустой.

**Результат** - таблица `INDEX`: одна строка = «колено X, ящик k, вот эта серия, в ней n файлов».
Её потом сохраняем как `index_partK.csv`.

In [ ]:
# Шесть ящиков: (имя, плоскость, Fat_Suppression). Номер ящика = номер в списке.
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]


def count_files(d):
    # сколько .dcm в папке серии = сколько в ней срезов. os.scandir быстрее, чем glob, на тысячах папок
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0                            # папки нет -> 0 срезов, такую серию не возьмём


def pick_series(series_df):
    # -> таблица «исследование, слот -> серия», одна строка на найденную комбинацию
    # series_df: строки train_series.csv (или test_series.csv): одна строка = одна серия
    df = series_df.copy()
    # путь к папке серии: <DATA_DIR>/train_series/<ID исследования>/<ID серии>
    df["dir"] = [str(DATA_DIR / "train_series" / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    # считаем файлы параллельно в SERIES_THREADS потоков: диск Kaggle медленный, потоки ждут его одновременно
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):          # g = все серии одного колена
        for k, (name, plane, fs) in enumerate(SLOTS):         # по 6 ящикам: k = 0..5
            # кандидаты в ящик k: нужная плоскость, нужный режим и хотя бы 3 среза (иначе это обзорный снимок)
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                r = cand.sort_values("n_files", ascending=False).iloc[0]   # серия с наибольшим числом срезов
                rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": name,
                             "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                             "plane": plane, "n_files": int(r["n_files"])})
    return pd.DataFrame(rows)

## Шаг 2. `read_stack`: одна серия -> одна стопка картинок

На входе папка с ~30 файлами `.dcm` в случайном порядке. На выходе массив `uint8 [n_срезов, 224, 224]`, срезы
по порядку в теле. Функция собрана из маленьких функций, по одной на шаг:

| # | Функция | Что делает | По-простому |
|---|---|---|---|
| 1 | `read_headers` | читает только заголовки всех файлов | открыли «паспорта» срезов, картинки не грузим (быстро) |
| 2 | `sort_slices` | сортирует по проекции `ImagePositionPatient` на нормаль к плоскости среза | разложили перемешанные ломтики батона обратно по порядку |
| 3 | `np.linspace` | если срезов больше `MAX_SLICES`, берёт 32 равномерно | из 320 ломтиков берём каждый десятый |
| 4 | `decode` | читает пиксели среза | теперь грузим саму картинку; битый файл заменяем соседним срезом |
| 5 | `canon_orient` | транспонирует/отражает по `ImageOrientationPatient` | у всех колен бедро сверху, перед колена в одну сторону |
| 6 | перцентили | 0.5% самых тёмных пикселей серии -> 0, 0.5% самых светлых -> 1 | у разных аппаратов разная шкала яркости; приводим к одной |
| 7 | `crop_resize` | вырезает 140×140 **мм** вокруг центра и сжимает до 224×224 | по `PixelSpacing` у всех колен одинаковый масштаб |
| 8 | `* 255 -> uint8` | 0..1 -> целые 0..255 | 1 байт на пиксель вместо 4 |

**Нормаль** - направление «сквозь» срезы. Её даёт векторное произведение направлений строк и столбцов картинки
(`np.cross`). Если спроецировать на неё положение каждого среза, получится «насколько далеко срез вдоль нарезки».
По этому числу и сортируем.

**Таблица `CANON`** задаёт, как должна выглядеть каждая плоскость: куда по картинке идёт перёд колена, где голова.
`canon_orient` смотрит, куда реально идут строки и столбцы снимка, и переворачивает его к этому виду.

In [ ]:
# Поля заголовка DICOM, которые нам нужны (читаем только их - так в разы быстрее):
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    # значение тега DICOM -> numpy-вектор из n чисел (или None, если тег пустой или битый)
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    # Шаг 1. Паспорта всех срезов серии БЕЗ картинок -> список словарей:
    # {"file": имя, "ipp": положение среза в теле (x,y,z мм), "iop": поворот (6 чисел),
    #  "ps": размер пикселя (мм по строкам и столбцам), "inst": номер среза от сканера}
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            # stop_before_pixels=True: картинку не читаем; specific_tags: только нужные поля
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass                            # битый заголовок: срез останется, но без координат
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # Шаг 2. Раскладываем срезы по порядку в теле.
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    # iop = 6 чисел: первые 3 - куда идут строки картинки, вторые 3 - куда идут столбцы (векторы в теле).
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        # нормаль = векторное произведение направлений строк и столбцов = направление «сквозь» срезы
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())   # вдоль какой оси тела режут: 0 сбоку, 1 спереди, 2 сверху
        if all(r["ipp"] is not None for r in recs):
            # ключ сортировки = проекция положения среза на нормаль = «насколько далеко срез вдоль нарезки».
            # sgn делает направление одинаковым у всех аппаратов (по таблице CANON)
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    # запасной путь: координат нет -> по номеру среза от сканера, потом по имени файла
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # Шаг 5. Поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps    # ascontiguousarray: после [::-1] массив «перевёрнут» в памяти


def crop_resize(img, ps):
    # Шаг 7. Квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями.
    # ps = (мм на пиксель по вертикали, по горизонтали). Пример: ps = 0.45 мм -> 140 мм = 311 пикселей.
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG               # 140 / 224 = 0.625 мм в пикселе результата
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))    # высота кропа в пикселях исходника (не больше картинки)
    cw = min(w, int(round(CROP_MM / ps[1])))    # ширина кропа
    y0, x0 = (h - ch) // 2, (w - cw) // 2       # левый верхний угол кропа: кроп по центру
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    # размер кропа в пикселях результата: ch пикселей * ps мм / 0.625 мм (если снимок меньше 140 мм - меньше 224)
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)     # cv2 принимает размер как (ширина, высота)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2                   # кладём в центр чёрного квадрата
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def decode(path):
    # Шаг 4. Файл .dcm -> картинка float32 [строки, столбцы] в «единицах сканера»
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    # RescaleSlope/Intercept: перевод сырых чисел в единицы сканера (у МРТ обычно 1 и 0)
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


def read_stack(d, plane_name):
    # серия -> uint8 [n_срезов, IMG, IMG] по порядку в теле, или None, если ничего не прочиталось
    recs = read_headers(d)                                  # шаг 1: паспорта
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)          # шаг 2: порядок
    n = len(recs)
    # шаг 3: не больше MAX_SLICES срезов, равномерно. n = 22 -> все 22; n = 320 -> 32 через ~10
    idx = np.unique(np.linspace(0, n - 1, min(n, MAX_SLICES)).round().astype(int))

    imgs = {}
    for i in idx:                                           # шаг 4: картинки
        try:
            imgs[int(i)] = decode(os.path.join(d, recs[int(i)]["file"]))
        except Exception:
            imgs[int(i)] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        return None
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:                                           # шаг 5: ориентация каждого среза
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        planes.append(canon_orient(a, iop, list(ps), ax))

    # шаг 6: яркость. Порог «чёрного» и «белого» - перцентили 0.5% и 99.5% по ВСЕЙ серии
    # (a[::4, ::4] - каждый 4-й пиксель: для перцентилей хватает, и в 16 раз быстрее)
    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [0.5, 99.5])
    # (a - lo) / (hi - lo) -> 0..1 (выбросы обрезаются clip), затем шаг 7: кроп 140 мм и 224 px
    stack = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    return (stack * 255).round().astype(np.uint8)          # шаг 8: 0..1 -> 0..255, 1 байт на пиксель

### Проверка глазами: что `read_stack` делает с одной серией

Берём одну сагиттальную серию и проходим шаги вручную:

1. **таблица «до»**: файлы в порядке, как они лежат в папке. Положение среза (мм от первого) скачет, `InstanceNumber` тоже бывает случайным;
2. **таблица «после»** `sort_slices`: положение растёт ровно от 0, шаг = расстояние между срезами (обычно 3-4 мм);
3. **картинки**: средний срез как есть в DICOM -> после ориентации -> после яркости, кропа 140 мм и 224 px;
4. **полоска**: 8 срезов из готовой стопки `read_stack`, от одного бока колена к другому.

На шаге 3 яркость считается по одному срезу, а в `read_stack` - по всей серии. Здесь это только для картинки.

In [ ]:
demo = series_demo[series_demo["Anatomical_Plane"] == "Sagittal"].iloc[0]       # первая сагиттальная серия
d = str(DATA_DIR / "train_series" / str(demo["StudyInstanceUID"]) / str(demo["SeriesInstanceUID"]))

recs = read_headers(d)                                    # шаг 1: паспорта в порядке файлов в папке
srt, ax, iop = sort_slices(list(recs), "Sagittal")        # шаг 2: по порядку в теле
normal = np.cross(iop[:3], iop[3:]) if iop is not None else None
sgn = CANON[ax][0] * np.sign(normal[ax]) if normal is not None else 1   # то же направление, что в sort_slices


def depth(r):
    # положение среза вдоль нарезки, мм = проекция ImagePositionPatient на нормаль (со знаком из CANON)
    return round(float(np.dot(r["ipp"], normal)) * sgn, 1) if normal is not None and r["ipp"] is not None else None


base = depth(srt[0]) or 0.0      # положение первого среза после сортировки: отсчитываем от него


def show_order(rs, title):
    print(title)
    display(pd.DataFrame([{"файл": r["file"][-12:], "InstanceNumber": r["inst"],
                           "мм от первого среза": None if depth(r) is None else round(depth(r) - base, 1)}
                          for r in rs[:8]]))


show_order(recs, f"ДО сортировки (как лежат в папке), первые 8 из {len(recs)}:")
show_order(srt, "ПОСЛЕ sort_slices:")

mid = srt[len(srt) // 2]                                  # средний срез серии
raw = decode(os.path.join(d, mid["file"]))                # шаг 4: картинка как есть
ps = mid["ps"] if mid["ps"] is not None else np.array([CROP_MM / max(raw.shape)] * 2)
oriented, ps2 = canon_orient(raw, iop, list(ps), ax)      # шаг 5: ориентация
lo, hi = np.percentile(oriented, [0.5, 99.5])             # шаг 6: яркость (здесь по одному срезу)
final = crop_resize(np.clip((oriented - lo) / max(hi - lo, 1e-6), 0, 1), ps2)   # шаг 7: 140 мм -> 224 px

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for a_, img, t in zip(axes, [raw, oriented, final],
                      [f"как в DICOM: {raw.shape[0]}x{raw.shape[1]} px, {ps[0]:.2f} мм/px",
                       "после canon_orient", f"после яркости и crop_resize: {IMG}x{IMG}, {CROP_MM / IMG:.3f} мм/px"]):
    a_.imshow(img, cmap="gray"); a_.set_title(t, fontsize=9); a_.axis("off")
plt.tight_layout(); plt.show()

stack = read_stack(d, "Sagittal")                         # всё сразу: готовая стопка
print(f"read_stack: {stack.shape} {stack.dtype}, яркость {stack.min()}..{stack.max()}")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for a_, j in zip(axes, np.linspace(0, len(stack) - 1, 8).round().astype(int)):
    a_.imshow(stack[j], cmap="gray"); a_.set_title(f"срез {j}", fontsize=8); a_.axis("off")
plt.tight_layout(); plt.show()

## Шаг 3. Сборка всех колен этой части

1. `studies[PART::N_PARTS]` - каждое второе колено по алфавиту ID, начиная с `PART`. Это деление только из-за
   лимита Output, на данные оно не влияет;
2. `pick_series` раскладывает серии этих колен по ящикам -> `INDEX`;
3. колена идут кусками по `CHUNK_STUDIES = 64`: серии куска читаются параллельно в 24 потока (`pool.map`);
4. стопки раскладываются в словарь словарей `by_study = {ID колена: {"slot0": стопка, "slot1": стопка, ...}}`;
5. каждое колено сохраняется в свой файл: `np.savez_compressed(studies/<ID>.npz, slot0=..., slot1=...)`.
   Ключи внутри файла - номера ящиков. Пустого ящика в файле просто нет;
6. в конце три файла-справки: `index_partK.csv` (`INDEX` + сколько срезов реально сохранено),
   `rsna_raw_meta.json` (с какими настройками собирали) и копия `train.csv`.

В логе после каждого куска: скорость (колен в секунду), сколько осталось минут и прогноз размера части.

In [ ]:
import shutil

train_df = pd.read_csv(DATA_DIR / "train.csv")                # одна строка = одно колено
train_series = pd.read_csv(DATA_DIR / "train_series.csv")     # одна строка = одна серия
studies = sorted(train_df["StudyInstanceUID"].astype(str))   # ID всех колен по алфавиту
part_studies = studies[PART::N_PARTS]                     # каждое N_PARTS-е исследование
series_part = train_series[train_series["StudyInstanceUID"].astype(str).isin(set(part_studies))]

INDEX = pick_series(series_part)                         # шаг 1 для всех колен части
INDEX["StudyInstanceUID"] = INDEX["StudyInstanceUID"].astype(str)
log(f"часть {PART}: {len(part_studies)} исследований, {len(INDEX)} серий в слотах")

# n_slices_saved: (ID, ящик) -> сколько срезов сохранили; failed: серии, которые не прочитались
t0, done, n_slices_saved, failed = time.time(), 0, {}, []
for c0 in range(0, len(part_studies), CHUNK_STUDIES):
    chunk = part_studies[c0:c0 + CHUNK_STUDIES]                    # 64 колена
    jobs = INDEX[INDEX["StudyInstanceUID"].isin(set(chunk))]       # их серии (~5 на колено)
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        # read_stack(dir, plane) для каждой строки jobs, 24 серии одновременно; порядок результатов = порядок jobs
        stacks = list(pool.map(read_stack, jobs["dir"], jobs["plane"]))
    by_study = {}
    for r, st in zip(jobs.itertuples(index=False), stacks):
        if st is None:
            failed.append(r.SeriesInstanceUID)
            continue
        # setdefault: если колена ещё нет в словаре, создать для него пустой словарь
        by_study.setdefault(r.StudyInstanceUID, {})[f"slot{r.slot}"] = st
        n_slices_saved[(r.StudyInstanceUID, r.slot)] = len(st)
    for study in chunk:
        # **{...}: словарь превращается в именованные аргументы slot0=..., slot2=...; файл сжимается (zip)
        np.savez_compressed(STUDY_DIR / f"{study}.npz", **by_study.get(study, {}))
    done += len(chunk)
    size_gb = sum(f.stat().st_size for f in STUDY_DIR.iterdir()) / 1024 ** 3
    rate = done / max(time.time() - t0, 1e-6)
    log(f"  {done}/{len(part_studies)} исследований, {rate:.2f} иссл/с, осталось ~{(len(part_studies) - done) / rate / 60:.0f} мин, "
        f"{size_gb:.2f} ГБ (прогноз на часть ~{size_gb / done * len(part_studies):.1f} ГБ)")

# файлы-справки
INDEX["n_slices_saved"] = [n_slices_saved.get((s, k), 0) for s, k in zip(INDEX["StudyInstanceUID"], INDEX["slot"])]
INDEX.drop(columns=["dir"]).to_csv(OUT_DIR / f"index_part{PART}.csv", index=False)   # путь dir на Kaggle локально не нужен
with open(OUT_DIR / "rsna_raw_meta.json", "w") as fh:
    json.dump({"IMG": IMG, "CROP_MM": CROP_MM, "MAX_SLICES": MAX_SLICES, "N_PARTS": N_PARTS,
               "SLOTS": [list(s) for s in SLOTS], "intensity": "percentile 0.5-99.5 per series -> uint8"}, fh, indent=1)
shutil.copy(DATA_DIR / "train.csv", OUT_DIR / "train.csv")
log(f"готово: {done} файлов, не прочитались {len(failed)} серий, "
    f"всего {sum(f.stat().st_size for f in STUDY_DIR.iterdir()) / 1024 ** 3:.2f} ГБ")

## Проверка глазами: готовый файл одного колена

Открываем один `.npz` из результата. `example.files` - список ключей (`slot0`, `slot2`, ...), по ключу - стопка
`[n_срезов, 224, 224]`. На картинке средний срез каждого найденного ящика и число срезов в нём.

In [ ]:
example = np.load(STUDY_DIR / f"{part_studies[len(part_studies) // 2]}.npz")   # колено из середины списка
keys = sorted(example.files)                     # например ["slot0", "slot1", "slot2", "slot3", "slot4"]
fig, axes = plt.subplots(1, len(keys), figsize=(3 * len(keys), 3.2))
for ax, key in zip(np.atleast_1d(axes), keys):  # atleast_1d: если ящик один, axes - не список, а одна ось
    st = example[key]                            # [n_срезов, 224, 224]
    ax.imshow(st[len(st) // 2], cmap="gray")     # средний срез
    ax.set_title(f"{SLOTS[int(key[4:])][0]}: {len(st)} срезов", fontsize=9)   # "slot3"[4:] = "3"
    ax.axis("off")
plt.tight_layout(); plt.show()

## Что дальше

1. Output этого запуска -> **New Dataset** (`rsna-knee-raw-part0` / `rsna-knee-raw-part1`).
2. `05_pack_cache_kaggle.ipynb` упаковывает часть в один файл `rsna_raw_partK.pack`: датасет из тысяч мелких
   файлов Kaggle скачивать не даёт.
3. Скачать оба `.pack` и распаковать в `data/raw/`: `tar -xf rsna_raw_part0.pack -C data/raw`.
4. Учить в `02_report_labels_mil.ipynb`.